In [ ]:
import numpy as np
import pandas as  pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [ ]:
warnings.formatwarning("ignore")

In [ ]:
data = pd.read_csv("data.csv")

In [ ]:
data.head()

In [ ]:
data.isna().sum()

In [ ]:
sns.countplot(x = data["class"])

In [ ]:
data.shape

In [ ]:
np.unique(data['class'], return_counts=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
for col in data.columns:
    if data[col].dtype == 'object':
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

# Z Scores

In [ ]:
from scipy.stats import zscore

In [ ]:
z_scores = zscore(data.drop(columns=['class']), axis = 0)
z_threshold = 2
np.abs(z_scores) > z_threshold

In [ ]:
y_pred_index = set (np.where(np.abs(z_scores) > 2)[0])
y_true_index = set (np.where(data['class'] == 1)[0])

In [ ]:
len(y_pred_index.intersection(y_true_index))

In [ ]:
len(y_pred_index)

In [ ]:
def zscore_outlier_detector(data, columns = None, threshold = 3):
    if columns == None:
        data_analyzer = data.drop(column=['class'])
    else:
        data_analyzer = data[columns]  


    z_scores = zscore(data_analyzer, axios=0)   
    y_pred_index = set(np.where(np.abs(z_scores) > threshold)[0]) 
    y_true_index = set(np.where(data['class'] == 0)[0])

    true_detection = y_pred_index.intersection(y_true_index)
    true_pos = len(true_detection) / len(y_true_index)
    true_neg = (len(y_pred_index) - len(true_detection)) / (len(data) - len(y_true_index))

    print(len(y_true_index))

    report = {"true_pos": true_pos, "true_neg": true_neg}
    return report


SyntaxError: expected ':' (2251746690.py, line 4)

In [ ]:
# select columns with best corrolation for getting best true_pos , true_neg

zscore_outlier_detector(data,columns= ["same_srv_rate", "dst_host_srv_count"],threshold=3)

In [ ]:
# select columns with more correlation

data.corr()['class'] > .9 


In [ ]:
corr_index = np.where ((data.corr()[["same_srv_rate", "dst_host_srv_count"]] > .9 ))

In [ ]:
data.column[corr_index]

# MAD for recognize outliers

In [ ]:
def robust_zscore_outlier_detector(data, columns = None, threshold = 3):
    if columns == None:
        data_analyzer = data.drop(column=['class'])
    else:
        data_analyzer = data[columns]  

    median = np.median(data_analyzer, axis = 0)
    MAD = np.median(np.abs(data_analyzer - median), axis = 0)
    mi = (.6745 * (data_analyzer - median)) / MAD

    y_pred_index = set(np.where(np.abs(mi) > threshold)[0]) 
    y_true_index = set(np.where(data['class'] == 0)[0])

    true_detection = y_pred_index.intersection(y_true_index)
    true_pos = len(true_detection) / len(y_true_index)
    true_neg = (len(y_pred_index) - len(true_detection)) / (len(data) - len(y_true_index))

    print(len(y_true_index))

    report = {"true_pos": true_pos, "true_neg": true_neg}
    return report



In [ ]:
# select columns with best corrolation for getting best true_pos , true_neg

robust_zscore_outlier_detector(data,columns = ["same_srv_rate", "dst_host_srv_count"], threshold=3.5)

In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
X = data.drop(columns='class')
y= data['class']

In [ ]:
scale =StandardScaler()
X = scale.fit_transform(X)

In [ ]:
clf = SVC()
clf.fit(X,y)

In [ ]:
clf.score(X,y)

In [ ]:
y_pred = clf.predict(X)

In [9]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y, y_pred))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.3, random_state=1234)

In [ ]:
clf = SVC()
clf.fit(X_train,y_train)

In [ ]:
y_pred_train = clf.predict(X_train)

In [ ]:
print(classification_report(y_train, y_pred_train))

In [ ]:
y_pred_test = clf.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred_test))